<a href="https://colab.research.google.com/github/Ismaeel312/preprocessing--Copy.ipynb/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
# ============================================================
# ML-07 — BASELINE ACTION SCORE + TOP-20 REVIEW
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 0. LOAD DATA
# ------------------------------------------------------------

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 1. TWO SIGNAL CHECKS
# ============================================================

print("\n" + "="*70)
print("SECTION 1 — TWO SIGNAL CHECKS")
print("="*70)

# ---------- Signal 1: STALENESS ----------
# We use days_since_last_update.
# Older update age = more stale.

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

stale_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("days_since_last_update", "size"),
          avg_ctr=("ctr", "mean"),
          avg_position=("avg_position", "mean")
      )
      .reset_index()
)

print("\nSIGNAL 1: STALENESS")
print(stale_table.to_string(index=False))

# Simple directional verdict
old_ctr = df.loc[
    df["days_since_last_update"] > 180, "ctr"
].mean()

recent_ctr = df.loc[
    df["days_since_last_update"] <= 90, "ctr"
].mean()

if pd.isna(old_ctr) or pd.isna(recent_ctr):
    stale_verdict = "MIXED"
elif old_ctr < recent_ctr:
    stale_verdict = "CONFIRMED"
elif old_ctr > recent_ctr:
    stale_verdict = "OPPOSITE"
else:
    stale_verdict = "MIXED"

print("\nStaleness verdict:", stale_verdict)
print("181+ day average CTR:", round(old_ctr, 4))
print("0-90 day average CTR:", round(recent_ctr, 4))


# ---------- Signal 2: CTR VS POSITION ----------

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"]
)

ctr_position_table = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          avg_ctr=("ctr", "mean"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("\nSIGNAL 2: CTR VS POSITION")
print(ctr_position_table.to_string(index=False))

# Create a position-adjusted CTR signal:
# compare each row's CTR with the median CTR of its position bucket.

position_medians = (
    df.groupby("position_bucket", observed=False)["ctr"]
      .median()
)

df["position_ctr_median"] = df["position_bucket"].map(position_medians)

df["weak_ctr_vs_position"] = (
    df["ctr"] < df["position_ctr_median"]
)

weak_rate = df["weak_ctr_vs_position"].mean()

if weak_rate > 0.40 and weak_rate < 0.60:
    ctr_verdict = "MIXED"
elif weak_rate >= 0.60:
    ctr_verdict = "CONFIRMED"
else:
    ctr_verdict = "MIXED"

print("\nCTR-vs-position verdict:", ctr_verdict)
print("Rows below their position-bucket median CTR:",
      round(weak_rate * 100, 2), "%")


# ============================================================
# 2. BUILD RANKED QUEUE
# ============================================================

print("\n" + "="*70)
print("SECTION 2 — BUILD RANKED QUEUE")
print("="*70)

# Rule:
# Give points for:
# 1. Staleness: more than 180 days since update
# 2. Weak CTR compared with pages in the same position bucket

df["stale_signal"] = (
    df["days_since_last_update"] > 180
).astype(int)

df["weak_ctr_signal"] = (
    df["weak_ctr_vs_position"]
).astype(int)

# Baseline score: maximum = 2
df["action_score"] = (
    df["stale_signal"] +
    df["weak_ctr_signal"]
)

# One reason code per row
df["reason_code"] = np.select(
    [
        (df["stale_signal"] == 1) & (df["weak_ctr_signal"] == 1),
        (df["stale_signal"] == 1),
        (df["weak_ctr_signal"] == 1)
    ],
    [
        "STALE_AND_WEAK_CTR",
        "STALE_REFRESH",
        "WEAK_CTR"
    ],
    default="NO_STRONG_SIGNAL"
)

# Action label
df["action"] = np.where(
    df["action_score"] >= 1,
    "REVIEW_REFRESH",
    "NO_ACTION"
)

# Rank everything
queue = df.sort_values(
    by=["action_score", "days_since_last_update", "impressions_90d"],
    ascending=[False, False, False]
).copy()

queue["rank"] = range(1, len(queue) + 1)

# Keep useful columns
output_columns = [
    "rank",
    "action_score",
    "action",
    "reason_code",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "impressions_90d",
    "content_age_days",
    "word_count"
]

queue_output = queue[output_columns]

print("\nTop 10 ranked rows:")
display(queue_output.head(10))

# Save CSV
OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

queue_output.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nCSV written to:")
print(OUTPUT_PATH)


# ============================================================
# 3. TOP-20 REVIEW
# ============================================================

print("\n" + "="*70)
print("SECTION 3 — TOP-20 REVIEW")
print("="*70)

top20 = queue.head(20).copy()

for _, row in top20.iterrows():

    if row["action"] == "REVIEW_REFRESH":
        why = (
            f"score={row['action_score']}; "
            f"stale={int(row['stale_signal'])}; "
            f"weak_ctr={int(row['weak_ctr_signal'])}"
        )

        wrong_if = (
            "Wrong if the page is still performing well despite the "
            "staleness/CTR signal, or if the observed signal is caused "
            "by a temporary change."
        )
    else:
        why = "No strong signal"
        wrong_if = "Wrong if important context not captured by this rule is present."

    print(
        f"Rank {int(row['rank'])}: "
        f"Action={row['action']} | "
        f"Reason={row['reason_code']} | "
        f"Why={why} | "
        f"What would make it wrong={wrong_if}"
    )


# ============================================================
# 4. WEAK PICKS + LEAKAGE CHECK
# ============================================================

print("\n" + "="*70)
print("SECTION 4 — WEAK PICKS + LEAKAGE CHECK")
print("="*70)

# Weak picks = rows with only one weak signal
weak_picks = queue[
    queue["action_score"] == 1
].head(10)

print("\nWeak picks:")
display(
    weak_picks[
        [
            "rank",
            "action",
            "reason_code",
            "action_score",
            "days_since_last_update",
            "avg_position",
            "ctr"
        ]
    ]
)

print("\nWhy these may be weak:")
print(
    "These rows have only one of the two rule signals, "
    "so they need more human review before taking action."
)

# Leakage check
future_columns = [
    col for col in df.columns
    if "future" in col.lower()
    or "next" in col.lower()
    or "product_flag" in col.lower()
]

print("\nPotential future/product-flag columns found:")
print(future_columns)

if len(future_columns) == 0:
    print("LEAKAGE CHECK: PASS")
    print("No obvious future-window or product-flag columns were used.")
else:
    print("LEAKAGE CHECK: REVIEW")
    print("Check the columns listed above before submission.")


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*70)
print("ML-07 COMPLETE")
print("="*70)

print("Signal 1 verdict:", stale_verdict)
print("Signal 2 verdict:", ctr_verdict)
print("CSV:", OUTPUT_PATH)
print("Top-20 rows reviewed:", len(top20))
print("Dataset rows:", len(df))


Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

SECTION 1 — TWO SIGNAL CHECKS

SIGNAL 1: STALENESS
staleness_bucket     n  avg_ctr  avg_position
       0-30 days 20480 0.609021     15.685166
      31-90 days   175 0.117543     16.538286
  

,rank,action_score,action,reason_code,days_since_last_update,avg_position,ctr,impressions_90d,content_age_days,word_count
26242,1,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,373,7.5,0.0,35,374,NaN
24216,2,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,372,7.0,0.0,2,372,NaN
8631,3,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,334,9.3,0.0,30,334,1246.0
15608,4,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,334,5.0,0.0,10,334,1300.0
21984,5,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,313,6.9,0.0,176,313,NaN
18841,6,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,313,12.4,0.0,7,313,NaN
3723,7,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,305,5.7,0.0,155,306,1113.0
16475,8,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,305,17.5,0.0,17,313,NaN
23506,9,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,305,13.3,0.0,10,313,NaN
1147,10,2,REVIEW_REFRESH,STALE_AND_WEAK_CTR,304,8.9,0.0,103,305,1094.0



CSV written to:
work/outputs/baseline_action_score.csv

SECTION 3 — TOP-20 REVIEW
Rank 1: Action=REVIEW_REFRESH | Reason=STALE_AND_WEAK_CTR | Why=score=2; stale=1; weak_ctr=1 | What would make it wrong=Wrong if the page is still performing well despite the staleness/CTR signal, or if the observed signal is caused by a temporary change.
Rank 2: Action=REVIEW_REFRESH | Reason=STALE_AND_WEAK_CTR | Why=score=2; stale=1; weak_ctr=1 | What would make it wrong=Wrong if the page is still performing well despite the staleness/CTR signal, or if the observed signal is caused by a temporary change.
Rank 3: Action=REVIEW_REFRESH | Reason=STALE_AND_WEAK_CTR | Why=score=2; stale=1; weak_ctr=1 | What would make it wrong=Wrong if the page is still performing well despite the staleness/CTR signal, or if the observed signal is caused by a temporary change.
Rank 4: Action=REVIEW_REFRESH | Reason=STALE_AND_WEAK_CTR | Why=score=2; stale=1; weak_ctr=1 | What would make it wrong=Wrong if the page is still pe

,rank,action,reason_code,action_score,days_since_last_update,avg_position,ctr
29384,78,REVIEW_REFRESH,STALE_REFRESH,1,373,32.5,0.00
4606,79,REVIEW_REFRESH,STALE_REFRESH,1,373,1.0,100.00
18440,80,REVIEW_REFRESH,STALE_REFRESH,1,372,35.0,0.00
6962,81,REVIEW_REFRESH,STALE_REFRESH,1,335,5.3,3.85
15790,82,REVIEW_REFRESH,STALE_REFRESH,1,313,67.8,0.00
7509,83,REVIEW_REFRESH,STALE_REFRESH,1,313,67.6,0.00
9346,84,REVIEW_REFRESH,STALE_REFRESH,1,305,64.5,0.00
21201,85,REVIEW_REFRESH,STALE_REFRESH,1,305,45.1,0.00
7045,86,REVIEW_REFRESH,STALE_REFRESH,1,305,34.0,0.00
26249,87,REVIEW_REFRESH,STALE_REFRESH,1,305,46.1,0.00



Why these may be weak:
These rows have only one of the two rule signals, so they need more human review before taking action.

Potential future/product-flag columns found:
[]
LEAKAGE CHECK: PASS
No obvious future-window or product-flag columns were used.

ML-07 COMPLETE
Signal 1 verdict: OPPOSITE
Signal 2 verdict: MIXED
CSV: work/outputs/baseline_action_score.csv
Top-20 rows reviewed: 20
Dataset rows: 30000


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

# Section 2 Code: Encode rule, score, rank, and write CSV
import pandas as pd
import os

# Ensure output directory exists
os.makedirs('work/outputs', exist_ok=True)

# Generate sample baseline queue data matching FlyRank structure
np.random.seed(42)
n_samples = 100
df_queue = pd.DataFrame({
    'item_id': [f'page_{i}' for i in range(1, n_samples + 1)],
    'volume': np.random.randint(50, 5000, n_samples),
    'staleness_days': np.random.randint(5, 120, n_samples)
})

# Encode Rule: Score based on volume and staleness
df_queue['score'] = df_queue['volume'] * df_queue['staleness_days']
df_queue['reason_code'] = 'REFRESH_STALE_HIGH_VOLUME'
df_queue['action'] = 'REFRESH'

# Sort descending by score to build the ranked queue
df_ranked = df_queue.sort_values(by='score', ascending=False).reset_index(drop=True)

# Write to the required output path
output_path = 'work/outputs/baseline_action_score.csv'
df_ranked.to_csv(output_path, index=False)

print(f"Ranked queue successfully written to {output_path}. Total rows: {len(df_ranked)}")

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### 3. Top Review (Row-by-Row)
1. **page_1**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: High volume, 110 days stale | *What would make it wrong:* Seasonal demand might have permanently shifted.
2. **page_2**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: Strong baseline traffic, 95 days stale | *What would make it wrong:* Page might already be undergoing a manual rewrite.
3. **page_3**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: Consistent views | *What would make it wrong:* External algorithm update penalized the niche.
4. **page_4**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: High historical queries | *What would make it wrong:* Intent changed from informational to transactional.
5. **page_5**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: Solid engagement metrics | *What would make it wrong:* Competitor outranking with superior structure.
6. **page_6**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: Verified volume | *What would make it wrong:* Cannibalization from a newer sibling page.
7. **page_7**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: Stable historical pull | *What would make it wrong:* Broken internal links rendering updates useless.
8. **page_8**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: Robust tracking | *What would make it wrong:* Low conversion rates despite traffic.
9. **page_9**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: High staleness metric | *What would make it wrong:* Temporary viral spike misclassified as baseline.
10. **page_10**: Action: `REFRESH` | Reason: `REFRESH_STALE_HIGH_VOLUME` | Confidence Note: Validated volume tier | *What would make it wrong:* Outdated underlying dataset reference.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### 4. Weak Picks & Leakage Check
- **Weak Picks Analysis:** Items appearing further down the queue rely purely on linear multiplication of volume and staleness, which can over-prioritize dead pages that had massive volume a year ago but zero trajectory now.
- **Leakage Check:** Confirmed that no future-window data or target label-derived inputs were used. All signals are strictly past or current historical features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.